<a href="https://colab.research.google.com/github/overwatch093-alt/Applied-Data-Science-1/blob/main/Tutorial_3_2_HDB_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# 1. DATA CLEANING FUNCTION
# ==========================================
def clean_hdb_dataset(raw_df):
    cleaned = raw_df.copy()

    # Issue 1: Remove Duplicate Rows (by txn_id)
    cleaned = cleaned.drop_duplicates(subset=['txn_id'], keep='first')

    # Issue 2: Standardize `month` column to YYYY-MM
    cleaned['month'] = pd.to_datetime(cleaned['month'], format='mixed').dt.strftime('%Y-%m')

    # Issue 3: Standardize `town` names
    town_map = {
        'AMK': 'ANG MO KIO',
        'Ang Mo Kio': 'ANG MO KIO',
        'ang mo kio': 'ANG MO KIO',
        'KALLANG WHAMPOA': 'KALLANG/WHAMPOA'
    }
    cleaned['town'] = cleaned['town'].astype(str).str.strip().replace(town_map)

    # Issue 4: Standardize `flat_type` categories
    flat_map = {
        '4-ROOM': '4 ROOM',
        '3 room': '3 ROOM',
        '4RM': '4 ROOM',
        'EXEC': 'EXECUTIVE'
    }
    cleaned['flat_type'] = cleaned['flat_type'].astype(str).str.strip().replace(flat_map)

    # Issue 5: Clean `resale_price` (remove '$', ',', fix 1000x scaled down values)
    cleaned['resale_price'] = (
        cleaned['resale_price']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    cleaned['resale_price'] = pd.to_numeric(cleaned['resale_price'], errors='coerce')
    cleaned.loc[cleaned['resale_price'] < 10000, 'resale_price'] *= 1000.0

    # Issue 6: Fix `floor_area_sqm` (scale back 10x inflated values & set 0 to NaN)
    cleaned.loc[cleaned['floor_area_sqm'] > 300, 'floor_area_sqm'] /= 10.0
    cleaned.loc[cleaned['floor_area_sqm'] <= 0, 'floor_area_sqm'] = np.nan

    # Issue 7: Fix `lease_commence_date` (set invalid years < 1960 or > 2024 to NaN)
    cleaned.loc[(cleaned['lease_commence_date'] < 1960) | (cleaned['lease_commence_date'] > 2024), 'lease_commence_date'] = np.nan

    # Issue 8 & 9: Replace invalid placeholder characters '-' and '?' with NaN
    cleaned['storey_range'] = cleaned['storey_range'].replace('-', np.nan)
    cleaned['flat_model'] = cleaned['flat_model'].replace('?', np.nan)

    # Feature Engineering: Extract transaction year
    cleaned['txn_year'] = pd.to_datetime(cleaned['month']).dt.year

    return cleaned

# ==========================================
# 2. LOAD DATA & APPLY CLEANING
# ==========================================
raw_df = pd.read_csv("Tutorial 3-2 HDB Dataset.csv")
df_cleaned = clean_hdb_dataset(raw_df)

# Define features and target
features = ['town', 'flat_type', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'txn_year']
X = df_cleaned[features]
y = df_cleaned['resale_price']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 3. BUILD LEAK-FREE PIPELINE
# ==========================================
num_cols = ['floor_area_sqm', 'lease_commence_date', 'txn_year']
cat_cols = ['town', 'flat_type', 'storey_range', 'flat_model']

# Numerical preprocessing (median imputation + scaling)
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing (frequent category imputation + one-hot encoding)
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Full Pipeline with Regressor
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit Pipeline and Predict
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# Evaluation Results
print(f"Model Test R² Score : {r2_score(y_test, y_pred):.4f}")
print(f"Model Test RMSE     : ${np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")

Model Test R² Score : 0.9123
Model Test RMSE     : $50,952.09
